# Gaming Laptop Uncleaned

In [ ]:
import pandas as pd
import re

In [ ]:
df = pd.read_csv("Gaming_laptop_uncleaned.csv")

# Basic cleaning and overlooking of the data

In [ ]:
print(df.to_string()) # print the summary statistics of the DataFrame

In [ ]:
print(df.head().T)  # print the first 5 rows of the DataFrame transposed for better readability

In [ ]:
# Set the maximum number of columns to display (None means unlimited)
pd.set_option('display.max_columns', None)

# Set the maximum width of the display in characters (None means unlimited)
pd.set_option('display.max_colwidth', None) # Prevents text truncation (...)
pd.set_option('display.width', None)        # Stretches the output

# If you are in Jupyter Notebook, you might also want this:
pd.set_option('display.expand_frame_repr', False) 

# Now print
print(df.head())

In [ ]:
print(df.describe())

In [ ]:
df.info()

In [ ]:
df.shape # rows, cols

In [ ]:
# Check missing values per column
df.isnull().sum()

In [ ]:
# Check for duplicate rows
print(df.duplicated().sum())

In [ ]:
# Standardize column names (remove spaces, convert to lowercase, replace spaces with underscores)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns)  # Print the standardized column names

# Starting the intermidiate cleaning of the data 

In [ ]:
df['price'] = df['price'].str.replace('₹', '').str.replace(',', '').astype(float)  # Convert price to float
df = df.rename(columns={'price_inr': 'price_INR(₹)'})  # Rename the price column to price_INR(₹)
df["specs_score"] = df["specs_score"].str.replace("Spec Score", "").astype(int)  # Convert specs_score to float
df['processor_brand'] = df['processor'].str.extract(r'(Intel|AMD)')# The parentheses in the regex tell pandas to extract that specific word
df = df[['name', 'specs_score', 'price','processor_brand', 'processor', 'core_threads', 'ram', 'storage', 'graphics_card', 'display_size', 'os', 'warranty']]
print(df.to_string())  # Print the first 5 values of the price column to verify conversion

In [ ]:
# Replace 'RTi' with ' Ti' and 'GTX' with ' GTX' 
df['graphics_card'] = df['graphics_card'].str.replace('RTi', ' Ti', regex=False)
df['graphics_card'] = df['graphics_card'].str.replace(r'(\d)Ti\b', r'\1 Ti', regex=True)
print(df.to_string())


In [ ]:
# Instead of filtering into separate dataframes, create a boolean column
df['is_touchscreen'] = df['display_size'].str.contains('Touch Screen', na=False)
print(df.to_string())


In [ ]:
# Extract VRAM
df['vram_gb'] = df['graphics_card'].str.extract(r'(\d+)\s*GB').astype(float)

# Extract GPU Brand (NVIDIA vs AMD)
df['gpu_brand'] = df['graphics_card'].str.extract(r'(NVIDIA|AMD|Intel)')

# Extract GPU Model (e.g., RTX 5070 Ti)
df['gpu_model'] = df['graphics_card'].str.extract(r'((?:RTX|GTX|Radeon|Iris Xe|Integrated)\s*\d+\s*(?:Ti|Super)?)')

print(df.to_string())


In [ ]:
# Extract the first word (Brand) from the laptop name
df['laptop_brand'] = df['name'].str.split().str[0].str.upper()

# Check the unique brands
print(df['laptop_brand'].value_counts())

In [ ]:
brands = ['LENOVO', 'ASUS', 'HP', 'MSI', 'ACER', 'ALIENWARE', 'DELL','GIGABYTE', 'RAZER', 'XIAOMI', 'RED Magic', 'ONEXPLAYER', 'HONOR']

pattern = r'\b(' + '|'.join(re.escape(b) for b in brands) + r')\b'
df['laptop_brand'] = df['name'].str.extract(pattern, flags=re.IGNORECASE)[0].str.upper()

print(df.to_string(index=False))

In [ ]:
df = df[['name','laptop_brand', 'specs_score', 'price', 'processor_brand', 'processor','core_threads', 'ram', 'storage', 'graphics_card', 'display_size', 'os','warranty', 'is_touchscreen', 'vram_gb', 'gpu_brand', 'gpu_model']]
df = df.sort_values(by='price', ascending=False)
print(df.to_string(index=False))

In [ ]:
# FIXING GPU MODEL NAMES
df['graphics_card'] = df['graphics_card'].str.replace("RTX 3030", "RTX 3080")
df['gpu_model'] = df['gpu_model'].str.replace("RTX 3030", "RTX 3080")
df['gpu_model'] = df['gpu_model'].str.replace("RTX3050", "RTX 3050")
print(df.to_string())

In [ ]:
df['ram'] = df['ram'].str.replace("16 GB  RAM", "16 GB RAM").str.replace("8 GB  RAM", "8 GB RAM").str.replace("32 GB  RAM", "32 GB RAM").str.replace("64 GB  RAM", "64 GB RAM").str.replace("12 GB  RAM", "12 GB RAM").str.replace("24 GB  RAM", "24 GB RAM")
print(df.to_string())

In [ ]:
df['storage'] = df['storage'].str.replace("8 GB SSD", "8 TB SSD").str.replace("1 GB SSD", "1 TB SSD")

In [ ]:
df['os'] = df['os'].str.replace("Windows 11  OS", "Windows 11 OS")

In [ ]:
os = df['vram_gb'].value_counts()
print(os.to_string())

In [ ]:
df.to_csv('laptops_cleaned.csv', index=False, encoding='utf-8-sig')

# AMD

In [ ]:
amd = df[df['processor_brand'] == 'AMD'].sort_values(by='price', ascending=False)  # Filter for AMD laptops and sort by price in descending order
print(amd.to_string())  # Print the first 5 rows of the AMD DataFrame to verify filtering


## AMD Touch Screen Models

In [ ]:
# Create a new dataframe with ONLY touch screen laptops
amd_touchScreen = amd[amd['display_size'].str.contains('Touch Screen', case=False, na=False)]
amd_touchScreen = amd_touchScreen.sort_values(by='price', ascending=False)  # Sort by price in descending order
print(amd_touchScreen.to_string())  # Print the first 5 rows of the AMD

# INTEL

In [ ]:
intel = df[df['processor_brand'] == 'Intel'].sort_values(by='price', ascending=False)  # Filter for Intel laptops and sort by price in descending order
print(intel.to_string())  # Print the first 5 rows of the Intel DataFrame to verify filtering

## INTEL Touch Screen Models

In [ ]:
# Create a new dataframe with ONLY touch screen laptops
intel_touchScreen = intel[intel['display_size'].str.contains('Touch Screen', case=False, na=False)]
intel_touchScreen = intel_touchScreen.sort_values(by='price', ascending=False)  # Sort by price in descending order
print(intel_touchScreen.to_string())  # Print the first 5 rows of the Intel touchscreen DataFrame to verify filtering